# deeptrack.backend.core

<a href="https://colab.research.google.com/github/DeepTrackAI/DeepTrack2/blob/develop/tutorials/3-advanced-topics/DTAT399A_backend.core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.

This advanced tutorial introduces the backend.core module.

## 1. What is `core`?

The `core` module provides fundamental utilities and functions to manage and process data on a low level.

In particular it provide tools to store, validate, and manage data and computational nodes with dependency tracking.


## 2. Basic Node Usage with Parent-Child Dependency

In [ ]:
from deeptrack.backend.core import DeepTrackNode

parent = DeepTrackNode(action=lambda: 10)
child = DeepTrackNode(action=lambda _ID=None: parent(_ID) * 2)

# Establish parent-child dependency.
parent.add_child(child)

# Store values.
parent.store(15, _ID=(0,))
parent.store(20, _ID=(1,))

# Compute values based on parent values.
child_value_0 = child(_ID=(0,))
child_value_1 = child(_ID=(1,))
print(child_value_0, child_value_1)

# Invalidate parent data for a given ID.
parent.invalidate((0,))
print(parent.is_valid((0,)))

# Update the parent value and recompute the child value:
print(child.is_valid((0,)))
parent.store(25, _ID=(0,))
child_value_recomputed = child(_ID=(0,))
print(child_value_recomputed)

30 40
False
False
50


## 3.  Lazy evaluation and Caching
Here we add a function to a `DeepTrackNode` which retuns a constant value and updates a global counter variable when called.

In [95]:
# Create counter node with side effect
call_count = 0
def calculation():
    global call_count
    call_count += 1
    return 10

node = DeepTrackNode(calculation)

# First call computes value.
print(node(), call_count) 

# Subsequent call uses cached value.
node.invalidate()
print(node(), call_count) 

# Invalidate and call again.
node.invalidate()
print(node(), call_count) 

10 1
10 2
10 3


## 4. Data Management with IDs

Map IDs to stored `DeepTrackData` objects lika a dictionary.

In [ ]:
from deeptrack.backend.core import DeepTrackDataDict

data_dict = DeepTrackDataDict()

# Create listings with unique indices.
data_dict.create_index((0, 0))
data_dict.create_index((0, 1))
data_dict.create_index((1, 0))
data_dict.create_index((1, 1))

# Store some data for the indices.
data_dict[(0, 0)].store("Cat")
data_dict[(0, 1)].store("Dog")
data_dict[(1, 0)].store("Mouse")
data_dict[(1, 1)].store("Bird")

# Print the indices.
print(data_dict[(0, 0)].current_value())
print(data_dict[(1, 1)].current_value())
print(data_dict[(0, )])

Cat
Bird
{(0, 0): <deeptrack.backend.core.DeepTrackDataObject object at 0x7f0dc3022b90>, (0, 1): <deeptrack.backend.core.DeepTrackDataObject object at 0x7f0dc302f790>}


## 5. Propagating operators
Nodes can also be used as simple handles for functions.

In [92]:
a = DeepTrackNode(lambda: 5 + 5)
b = DeepTrackNode(lambda: 3 + 3)

sum_node = a + b
product_node = a * b

print(sum_node())
print(product_node())

16
60


## 6. Validation control
Validate or invalidate nodes manually to enable/disable storing data.

In [88]:
node = DeepTrackNode(lambda: 42)
node.store(100)

print(node())

# Validate.
node.validate()
print(node.is_valid())
print(node()) 

# Invalidate.
node.invalidate()
print(node.is_valid())
print(node())

100
True
100
False
42


## 7. Get Citations
The `DeepTrackNode` class can also be used to obtain citations.

In [101]:
DeepTrackNode().get_citations()

{'\n@article{Midtvet2021DeepTrack,\n    author  = {Midtvedt,Benjamin  and \n               Helgadottir,Saga  and \n               Argun,Aykut  and \n               Pineda,Jesús  and \n               Midtvedt,Daniel  and \n               Volpe,Giovanni},\n    title   = {Quantitative digital microscopy with deep learning},\n    journal = {Applied Physics Reviews},\n    volume  = {8},\n    number  = {1},\n    pages   = {011310},\n    year    = {2021},\n    doi     = {10.1063/5.0034891}\n}\n'}